# Batch cell typing – IHOPE project

Runs normalization, AnnData construction, GMM thresholding, rule-based cell typing, and summary export for all 13 samples. Starts from pre-filtered CSVs. Paths are built with pathlib so this runs the same on Windows or Mac.

In [ ]:
import sys
import gc
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed"
ANNDATA_DIR = DATA_DIR / "anndata"
ANNDATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from scripts.transforms import apply_transform
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad
from scripts.annotation import compute_positivity_matrix
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

## Sample list

`file_basename` plus `input_suffix` is the name used to find the input CSV on disk. `out_basename` is the name used for all outputs.

In [ ]:
# (file_basename, input_suffix, out_basename)
SAMPLES = [
    ("IHOPE14_MedLN_BottomLeft",  "_cleaned_filtered",        "IHOPE14_MedLN_BottomLeft"),
    ("IHOPE14_MedLN_TopRight",    "_cleaned_filtered_looped", "IHOPE14_MedLN_TopRight"),
    ("IHOPE14_MedLN_BottomRight", "_cleaned_filtered_looped", "IHOPE14_MedLN_BottomRight"),
    ("IHOPE14_mesLN",             "_cleaned_filtered_looped", "IHOPE14_MesLN"),
    ("IHOPE20_LN",                "_cleaned_filtered",        "IHOPE20_MedLN"),
    ("IHOPE20_Spleen",            "_cleaned_filtered_looped", "IHOPE20_Spleen"),
    ("IHOPE26_LN",                "_cleaned_filtered",        "IHOPE26_MedLN"),
    ("IHOPE26_Spleen",            "_cleaned_filtered",        "IHOPE26_Spleen"),
    ("IHOPE27_LN",                "_cleaned_filtered",        "IHOPE27_MedLN"),
    ("IHOPE27_Spleen",            "_cleaned_filtered",        "IHOPE27_Spleen"),
    ("IHOPE39_LN",                "_cleaned_filtered",        "IHOPE39_MedLN"),
    ("IHOPE39_MesLN_A",           "_cleaned_filtered",        "IHOPE39_MesLN_A"),
    ("IHOPE39_MesLN_B",           "_cleaned_filtered",        "IHOPE39_MesLN_B"),
    ("IHOPE39_Spleen",            "_cleaned_filtered",        "IHOPE39_Spleen"),
]

## Batch pipeline

For each sample: normalize from the cleaned filtered CSV, build AnnData, run GMM thresholding, assign cell types, and export a summary CSV.

In [ ]:
failed = []

for file_basename, input_suffix, out_basename in SAMPLES:
    print(f"Processing: {out_basename}")

    try:
        # Normalize
        input_file  = DATA_DIR / f"{file_basename}{input_suffix}.csv"
        zscore_file = DATA_DIR / f"{out_basename}_cleaned_filtered_zscore.csv"

        df_out, markers, metadata, fig_raw, fig_normalized  = apply_transform(
            input_file=str(input_file),
            method="zscore",
            output_file=str(zscore_file),
            save_plot=True,
        )
        plt.close(fig_raw)
        plt.close(fig_normalized)

        print(f"  Normalized: {len(df_out)} cells, {len(markers)} markers")

        # Build AnnData
        adata = load_and_build_anndata(str(zscore_file))
        del df_out
        gc.collect()
        print(f"  AnnData built: {adata.shape}")

        # GMM thresholding
        adata, thresholds, best_gmms = compute_positivity_matrix(
            adata,
            quantile=0.8,
            random_state=0,
        )
        print(f"  GMM thresholding done")

        # Save AnnData after GMM
        h5ad_gmm = ANNDATA_DIR / f"{out_basename}_filtered_zscore_GMM.h5ad"
        save_h5ad(adata, str(h5ad_gmm))
        print(f"  Saved: {h5ad_gmm}")

        # Rule-based cell typing
        adata = assign_cell_types_bool_IHOPE(adata)
        print(f"  Cell types assigned")

        # Save AnnData with cell types
        h5ad_typed = ANNDATA_DIR / f"{out_basename}_filtered_zscore_GMM_IHOPE_celltypes.h5ad"
        save_h5ad(adata, str(h5ad_typed))
        print(f"  Saved: {h5ad_typed}")

        # Summary CSV
        df_summary = summarize_celltypes_IHOPE(
            adata,
            filename=f"{out_basename}_filtered_zscore_IHOPE_summary.csv",
        )
        print(f"  Summary exported")

        del adata
        gc.collect()

    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append((out_basename, str(e)))

print("Batch complete.")
if failed:
    print(f"Failed samples ({len(failed)}):")
    for name, err in failed:
        print(f"  {name}: {err}")
else:
    print("All samples processed.")